First, we need to set up the environment with the necessary Hugging Face libraries, including bitsandbytes for memory-efficient model loading and llmlingua for the base token importance.

In [1]:
!pip install -q transformers datasets accelerate bitsandbytes torch
!pip install -q llmlingua

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00


We will load the TweetEval dataset, the RoBERTa classifier, and the Qwen2.5 generator.

In [2]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification, BitsAndBytesConfig
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Load Dataset (TweetEval - Irony)
print("Loading Dataset...")
dataset = load_dataset("tweet_eval", "irony")
test_data = dataset['test']

# 2. Load Evaluation Classifier (RoBERTa)
print("Loading RoBERTa Classifier...")
roberta_name = "cardiffnlp/twitter-roberta-base-irony"
rob_tokenizer = AutoTokenizer.from_pretrained(roberta_name)
rob_model = AutoModelForSequenceClassification.from_pretrained(roberta_name).to(device)

# 3. Define the 4-bit Quantization Configuration
print("Configuring 4-bit Quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16 # Speeds up computation on T4
)

# 4. Load Reasoning Generator (Qwen 3B with updated quantization)
print("Loading Qwen2.5-3B...")
qwen_name = "Qwen/Qwen2.5-3B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_name,
    device_map="auto",
    quantization_config=bnb_config
)

Loading Dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

irony/train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

irony/test-00000-of-00001.parquet:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

This cell translates our equation for $H(c)$ and the scaled sigmoid transformation into code. It extracts the logits from Qwen's generation to calculate how "uncertain" or complex the reasoning is.

In [3]:
import torch
import torch.nn.functional as F
import math

def calculate_dynamic_gamma(logits, min_gamma=0.1, max_gamma=0.9, k=1.0, mu=0.5):
    """
    Calculates the reasoning entropy and maps it to a dynamic compression ratio.
    """
    # Convert logits to probabilities
    probs = F.softmax(logits, dim=-1)

    # Epsilon (1e-9) to prevent log(0) yielding -inf and breaking the sum into a nan
    epsilon = 1e-9
    entropy_tensor = -torch.sum(probs * torch.log(probs + epsilon), dim=-1)
    entropy = entropy_tensor.mean().item()

    # Fallback catch just in case a nan slips through due to precision limits
    if math.isnan(entropy):
        entropy = 0.0

    # USING math.exp instead of np.exp since entropy is a standard float here
    gamma_dyn = min_gamma + (max_gamma - min_gamma) / (1 + math.exp(-k * (entropy - mu)))

    return gamma_dyn, entropy

This is the core of the adversarial robustness. We pass the text through RoBERTa, grab the embeddings, and compute the gradient of the loss with respect to those embeddings.

In [4]:
def get_gradient_sensitivity(text, rob_tokenizer, rob_model, device, target_class=1):
    """
    Approximates token impact by checking the gradient norm of the embeddings
    during a backward pass in the RoBERTa classifier.
    """
    inputs = rob_tokenizer(text, return_tensors="pt").to(device)

    # Get the embedding layer output and retain its gradients
    embeddings = rob_model.roberta.embeddings.word_embeddings(inputs['input_ids'])
    embeddings.retain_grad()

    # Forward pass
    outputs = rob_model(inputs_embeds=embeddings, attention_mask=inputs['attention_mask'])
    logits = outputs.logits

    # Compute a dummy loss against the predicted or target class
    loss = F.cross_entropy(logits, torch.tensor([target_class]).to(device))

    # Backward pass to get gradients
    rob_model.zero_grad()
    loss.backward()

    # Calculate the L2 norm of the gradients for each token
    grad_norms = embeddings.grad.norm(dim=-1).squeeze().cpu().numpy()

    # Map back to tokens
    tokens = rob_tokenizer.convert_ids_to_tokens(inputs['input_ids'].squeeze())

    # Explicit list of tokens and punctuation to completely ignore
    skip_tokens = {'<s>', '</s>', '<pad>', '<unk>', '.', ',', ':', '"', "'", '?', '!'}

    sensitivity_scores = {}
    for token, grad in zip(tokens, grad_norms):
        # Clean special characters from RoBERTa tokens (like Ġ)
        clean_token = token.replace('Ġ', '').strip()

        # Filtering logic to ignore special tokens and single-character noise
        if clean_token and token not in skip_tokens and len(clean_token) > 1:
            # If a word is split into subwords, keep the max gradient value
            if clean_token in sensitivity_scores:
                sensitivity_scores[clean_token] = max(sensitivity_scores[clean_token], float(grad))
            else:
                sensitivity_scores[clean_token] = float(grad)

    return sensitivity_scores

Ablation

In [5]:
import torch
import torch.nn.functional as F
import nltk, re
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

def classify_tweet(tweet_text, rob_tokenizer, rob_model, device):
    """Classify the raw tweet and return (predicted_class, logits_tensor)."""
    inputs = rob_tokenizer(tweet_text, return_tensors="pt",
                           truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = rob_model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
    return prediction, logits


# ─────────────────────────────────────────────────────────────────────────────
# ROOT CAUSE FIX: The old keyword scanner gave cot_signal=1.0 for almost every
# tweet because Qwen uses words like "ironic" and "sarcasm" even when describing
# WHY something is NOT ironic. This made the CoT signal useless.
#
# NEW APPROACH: Parse only the VERDICT LINE that Qwen is prompted to output at
# the end of its reasoning ("Verdict: IRONIC" / "Verdict: NON-IRONIC").
# This is a hard 0/1 signal from Qwen's actual conclusion, not its vocabulary.
#
# Fallback: if the verdict line is missing, scan only the FINAL 2 SENTENCES
# (the conclusion), not the full CoT body where analytical vocabulary pollutes.
# ─────────────────────────────────────────────────────────────────────────────
def extract_structured_cot_signal(cot_text):
    """
    Returns a float in {0.0, 0.5, 1.0}:
      1.0 → Qwen explicitly concluded IRONIC
      0.0 → Qwen explicitly concluded NON-IRONIC
      0.5 → No clear verdict found (neutral / abstain)
    """
    lower = cot_text.lower()

    # Priority 1: explicit verdict line
    verdict_match = re.search(
        r"verdict\s*[:\-]\s*(ironic|non[- ]?ironic|not ironic|sarcastic|not sarcastic)",
        lower
    )
    if verdict_match:
        v = verdict_match.group(1)
        if "non" in v or "not" in v:
            return 0.0
        else:
            return 1.0

    # Priority 2: scan only the final 2 sentences
    sentences = re.split(r"(?<=[.!?])\s+", cot_text.strip())
    conclusion = " ".join(sentences[-2:]).lower() if len(sentences) >= 2 else lower

    CONCLUDE_IRONIC   = {"is ironic", "is sarcastic", "contains irony",
                         "irony is present", "this is ironic", "therefore ironic",
                         "tweet is ironic", "concludes ironic"}
    CONCLUDE_NONIRONIC = {"is not ironic", "is not sarcastic", "no irony",
                          "not ironic", "non-ironic", "sincere", "literal statement",
                          "therefore not", "no sarcasm", "not sarcastic"}

    ironic_hits    = sum(1 for p in CONCLUDE_IRONIC    if p in conclusion)
    nonironic_hits = sum(1 for p in CONCLUDE_NONIRONIC if p in conclusion)

    if ironic_hits > nonironic_hits:    return 1.0
    if nonironic_hits > ironic_hits:    return 0.0
    return 0.5   # genuinely ambiguous conclusion


# ─────────────────────────────────────────────────────────────────────────────
# entropy_to_alpha  — the entropy-gated adaptive fusion weight
# KEY INSIGHT (from ablation analysis):
#   ENTROPY_ONLY outperforms FULL_RDS because the CoT signal was too noisy.
#   But entropy itself IS informative:
#     Low H(c)  → Qwen generated confidently → its verdict is more trustworthy
#                 → reduce alpha (give CoT more weight)
#     High H(c) → Qwen was uncertain during generation → verdict less reliable
#                 → raise alpha (lean on RoBERTa more)
# This replaces the fixed alpha=0.7 with a function of entropy.
# ─────────────────────────────────────────────────────────────────────────────
def entropy_to_alpha(entropy,
                     low_thresh=0.2, high_thresh=0.5,
                     alpha_low=0.55,  alpha_high=0.85):
    """
    Maps reasoning entropy to a RoBERTa weight (alpha).
      entropy <= low_thresh  → alpha = alpha_low  (trust CoT more)
      entropy >= high_thresh → alpha = alpha_high (trust RoBERTa more)
      in between             → linear interpolation
    """
    if entropy <= low_thresh:
        return alpha_low
    elif entropy >= high_thresh:
        return alpha_high
    else:
        t = (entropy - low_thresh) / (high_thresh - low_thresh)
        return alpha_low + t * (alpha_high - alpha_low)


# ─────────────────────────────────────────────────────────────────────────────
# fused_classify_v2  — the improved decision function

# Formula:
#   alpha = entropy_to_alpha(H(c))       ← dynamic, not fixed
#   p_final = alpha * p_RoBERTa + (1-alpha) * p_CoT_verdict
# ─────────────────────────────────────────────────────────────────────────────
def fused_classify_v2(tweet_text, cot_text, entropy,
                      rob_tokenizer, rob_model, device):
    """
    Entropy-gated fusion of RoBERTa tweet probability and Qwen verdict signal.
    Returns: (prediction, roberta_prob, cot_signal, alpha, fused_prob)
    """
    # RoBERTa on the original tweet
    _, tweet_logits = classify_tweet(tweet_text, rob_tokenizer, rob_model, device)
    tweet_probs = F.softmax(tweet_logits, dim=-1).squeeze().cpu()
    p_roberta = tweet_probs[1].item()   # P(ironic) from RoBERTa

    # Qwen structured verdict
    cot_signal = extract_structured_cot_signal(cot_text)

    # Entropy-adaptive alpha
    alpha = entropy_to_alpha(entropy)

    # Weighted fusion
    p_fused = alpha * p_roberta + (1 - alpha) * cot_signal
    prediction = int(p_fused >= 0.5)

    return prediction, p_roberta, cot_signal, alpha, p_fused


# ─────────────────────────────────────────────────────────────────────────────
# improved compress_cot  (contrastive connectors preserved)
# ─────────────────────────────────────────────────────────────────────────────
CONTRASTIVE_CONNECTORS = {
    "but", "yet", "though", "although", "however", "despite", "while",
    "whereas", "still", "even", "never", "not", "no", "without", "barely"
}

def compress_cot_improved(original_text, gamma_dyn, critical_tokens):
    stop_words = set(stopwords.words("english")) - CONTRASTIVE_CONNECTORS
    words = original_text.split()
    target_length = max(5, int(len(words) * (1.0 - gamma_dyn)))
    scored_words = []
    for idx, word in enumerate(words):
        clean = word.lower().strip(".,!?\"'")
        is_critical = any(crit.lower() in clean for crit in critical_tokens if len(crit) >= 4)
        if is_critical:                              score = 1000
        elif clean in CONTRASTIVE_CONNECTORS:        score = 500
        elif clean not in stop_words and len(clean) >= 3: score = 10
        else:                                        score = 1
        scored_words.append((idx, word, score))
    scored_words.sort(key=lambda x: x[2], reverse=True)
    kept = sorted(scored_words[:target_length], key=lambda x: x[0])
    return " ".join(w[1] for w in kept)

print("✅ All upgraded helper functions loaded.")
print("   • extract_structured_cot_signal  — verdict-line parser (fixes noisy CoT signal)")
print("   • entropy_to_alpha               — entropy-gated adaptive fusion weight")
print("   • fused_classify_v2              — improved decision function")

✅ All upgraded helper functions loaded.
   • extract_structured_cot_signal  — verdict-line parser (fixes noisy CoT signal)
   • entropy_to_alpha               — entropy-gated adaptive fusion weight
   • fused_classify_v2              — improved decision function


This cell loops through a few test examples. It prompts Qwen for a Chain-of-Thought, calculates the dynamic budget, extracts the gradient sensitivities, and prepares the data for the final fusion.

In [10]:
import torch
import pandas as pd
from IPython.display import display

# ────────────────────────────────────────────────────────────────
# IMPROVED MAIN PIPELINE  —  RDS v2
#
# Three targeted upgrades over the previous version:
#
#   [1] Structured CoT verdict prompt
#       Qwen is now asked to end its reasoning with an explicit "Verdict: IRONIC"
#       or "Verdict: NON-IRONIC" line. This replaces the noisy keyword scan that
#       was returning signal=1.0 for almost every tweet.
#
#   [2] Entropy-gated adaptive alpha
#       Alpha is no longer fixed at 0.7. Low entropy (confident Qwen) → lower
#       alpha (trust Qwen more). High entropy (uncertain Qwen) → higher alpha
#       (trust RoBERTa more). Derived from the ablation study showing
#       ENTROPY_ONLY beat FULL_RDS because the CoT signal was corrupted.
#
#   [3] Fusion on p_ironic directly
#       No change to the fusion arithmetic — just cleaner inputs.
# ────────────────────────────────────────────────────────────────

STRUCTURED_PROMPT = (
    "Analyze whether the following tweet is ironic or non-ironic. "
    "Think step-by-step, then end your response with EXACTLY one of:\n"
    "Verdict: IRONIC\n"
    "Verdict: NON-IRONIC\n\n"
    "Tweet: \'{tweet}\'"
)

def generate_cot_and_analyze_v2(tweet, mode="FULL_RDS", tweet_num=""):
    num_str = f" {tweet_num}" if tweet_num else ""
    print(f"\n--- Processing Tweet{num_str}: {tweet[:60]}... ---")

    prompt = STRUCTURED_PROMPT.format(tweet=tweet)
    inputs = qwen_tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = qwen_model.generate(
            **inputs,
            max_new_tokens=120,
            return_dict_in_generate=True,
            output_scores=True
        )

    generated_ids = outputs.sequences[0][inputs["input_ids"].shape[1]:]
    cot_text = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True)
    print(f"  CoT Length : {len(cot_text.split())} words")

    if mode in ["FULL_RDS", "ENTROPY_ONLY"]:
        logits_stack = torch.stack(outputs.scores, dim=1).squeeze(0)
        gamma_dyn, entropy = calculate_dynamic_gamma(logits_stack)
        print(f"  Entropy    : {entropy:.3f}  →  Gamma: {gamma_dyn:.3f}")
    else:
        gamma_dyn, entropy = 0.5, 0.0

    if mode in ["FULL_RDS", "GRADIENT_ONLY"]:
        sensitivities = get_gradient_sensitivity(
            cot_text, rob_tokenizer, rob_model, device, target_class=1
        )
        sorted_sens = sorted(sensitivities.items(), key=lambda x: x[1], reverse=True)
        critical_tokens = [t[0] for t in sorted_sens if len(t[0]) >= 4][:5]
        print(f"  Crit.Tokens: {critical_tokens}")
    else:
        critical_tokens = []

    return {
        "tweet": tweet,
        "cot_text": cot_text,
        "entropy": entropy,
        "gamma_dyn": gamma_dyn,
        "critical_tokens": critical_tokens,
        "mode": mode
    }

Ablation Study (Entropy Only and Gradient only)

Entropy Only

In [11]:
import torch
import pandas as pd
from IPython.display import display

CURRENT_MODE = "ENTROPY_ONLY"
NUM_SAMPLES  = 100

results_list        = []
correct_predictions = 0

print(f"🚀 Starting RDS v2 Pipeline in {CURRENT_MODE} mode...")
print(f"   Upgrades: structured verdict prompt + entropy-adaptive alpha\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # 1. Generate CoT + entropy (no gradient step in ENTROPY_ONLY)
    result_dict = generate_cot_and_analyze_v2(sample_tweet, mode=CURRENT_MODE, tweet_num=i+1)

    # 2. Compress CoT FIRST using the dynamic budget + critical-token protection
    compressed_text = compress_cot_improved(
        result_dict["cot_text"],
        result_dict["gamma_dyn"],
        result_dict["critical_tokens"]
    )

    # 3. THEN classify using the COMPRESSED CoT
    prediction, p_roberta, cot_signal, alpha, p_fused = fused_classify_v2(
        sample_tweet,
        compressed_text,            # ← was result_dict["cot_text"]
        result_dict["entropy"],
        rob_tokenizer, rob_model, device
    )

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  RoBERTa={p_roberta:.3f} | CoT={cot_signal:.1f} | α={alpha:.3f} "
          f"| Fused={p_fused:.3f} | Pred={prediction} | True={true_label} "
          f"| {'✅' if is_correct else '❌'}")

    results_list.append({
        "Tweet"             : sample_tweet[:35] + "...",
        "True"              : true_label,
        "Pred"              : prediction,
        "✓?"                : "✅" if is_correct else "❌",
        "RoBERTa P(ironic)" : round(p_roberta, 3),
        "CoT Verdict"       : cot_signal,
        "α (adaptive)"      : round(alpha, 3),
        "Fused P(ironic)"   : round(p_fused, 3),
        "Entropy"           : round(result_dict["entropy"], 3),
        "Gamma"             : round(result_dict["gamma_dyn"], 3),
        "Orig Len"          : len(result_dict["cot_text"].split()),
        "Comp Len"          : len(compressed_text.split()),
    })

accuracy_pct = (correct_predictions / NUM_SAMPLES) * 100
skip_ratio   = sum(1 - r["Comp Len"]/r["Orig Len"] for r in results_list) / NUM_SAMPLES

print(f"\n{'='*64}")
print(f"🏆 RDS v2 [{CURRENT_MODE}] ACCURACY : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📉 Avg Token-Skip Ratio         : {skip_ratio:.1%}")
print(f"⚖️  Alpha range used             : [0.35 – 0.75] (entropy-gated)")
print(f"{'='*64}")

df = pd.DataFrame(results_list)
display(df)

🚀 Starting RDS v2 Pipeline in ENTROPY_ONLY mode...
   Upgrades: structured verdict prompt + entropy-adaptive alpha


--- Processing Tweet 1: @user Can U Help?||More conservatives needed on #TSU + get p... ---
  CoT Length : 86 words
  Entropy    : 0.339  →  Gamma: 0.468
  RoBERTa=0.303 | CoT=1.0 | α=0.689 | Fused=0.520 | Pred=1 | True=0 | ❌

--- Processing Tweet 2: Just walked in to #Starbucks and asked for a "tall blonde" H... ---
  CoT Length : 90 words
  Entropy    : 0.552  →  Gamma: 0.510
  RoBERTa=0.650 | CoT=1.0 | α=0.850 | Fused=0.702 | Pred=1 | True=1 | ✅

--- Processing Tweet 3: #NOT GONNA WIN... ---
  CoT Length : 93 words
  Entropy    : 0.533  →  Gamma: 0.507
  RoBERTa=0.033 | CoT=0.5 | α=0.850 | Fused=0.103 | Pred=0 | True=0 | ✅

--- Processing Tweet 4: @user He is exactly that sort of person. Weirdo!... ---
  CoT Length : 93 words
  Entropy    : 0.213  →  Gamma: 0.443
  RoBERTa=0.209 | CoT=0.5 | α=0.563 | Fused=0.336 | Pred=0 | True=0 | ✅

--- Processing Tweet 5: So much #

,Tweet,True,Pred,✓?,RoBERTa P(ironic),CoT Verdict,α (adaptive),Fused P(ironic),Entropy,Gamma,Orig Len,Comp Len
0,@user Can U Help?||More conservativ...,0,1,❌,0.303,1.0,0.689,0.520,0.339,0.468,86,45
1,Just walked in to #Starbucks and as...,1,1,✅,0.650,1.0,0.850,0.702,0.552,0.510,90,44
2,#NOT GONNA WIN...,0,0,✅,0.033,0.5,0.850,0.103,0.533,0.507,93,45
3,@user He is exactly that sort of pe...,0,0,✅,0.209,0.5,0.563,0.336,0.213,0.443,93,51
4,So much #sarcasm at work mate 10/10...,1,0,❌,0.075,1.0,0.806,0.254,0.456,0.491,83,42
...,...,...,...,...,...,...,...,...,...,...,...,...
95,The last day of the year and I will...,0,0,✅,0.133,1.0,0.747,0.352,0.397,0.479,84,43
96,MB forgot to move the elf on shelf ...,0,1,❌,0.629,1.0,0.710,0.737,0.360,0.472,84,44
97,"@user happy new year, da! Thanks so...",0,0,✅,0.095,0.5,0.748,0.198,0.398,0.480,99,51
98,WATCH: #Megatron from #Transformers...,0,1,❌,0.515,0.5,0.722,0.511,0.372,0.474,87,45


Gradient Only

In [12]:
import torch
import pandas as pd
from IPython.display import display

CURRENT_MODE = "GRADIENT_ONLY"
NUM_SAMPLES  = 100

results_list        = []
correct_predictions = 0

print(f"🚀 Starting RDS v2 Pipeline in {CURRENT_MODE} mode...")
print(f"   Upgrades: structured verdict prompt + entropy-adaptive alpha\n")

for i in range(NUM_SAMPLES):
    sample_tweet = test_data[i]["text"]
    true_label   = test_data[i]["label"]

    # 1. Generate CoT + gradient step (no entropy in GRADIENT_ONLY)
    result_dict = generate_cot_and_analyze_v2(sample_tweet, mode=CURRENT_MODE)

    # 2. Compress CoT FIRST using the dynamic budget + critical-token protection
    #    Note: entropy=0.0 in GRADIENT_ONLY → alpha defaults to alpha_low (0.35)
    #    meaning CoT verdict gets maximum weight in this mode
    compressed_text = compress_cot_improved(
        result_dict["cot_text"],
        result_dict["gamma_dyn"],
        result_dict["critical_tokens"]
    )

    # 3. THEN classify using the COMPRESSED CoT
    prediction, p_roberta, cot_signal, alpha, p_fused = fused_classify_v2(
        sample_tweet,
        compressed_text,            # ← was result_dict["cot_text"]
        result_dict["entropy"],
        rob_tokenizer, rob_model, device
    )

    is_correct = (prediction == true_label)
    if is_correct:
        correct_predictions += 1

    print(f"  RoBERTa={p_roberta:.3f} | CoT={cot_signal:.1f} | α={alpha:.3f} "
          f"| Fused={p_fused:.3f} | Pred={prediction} | True={true_label} "
          f"| {'✅' if is_correct else '❌'}")

    results_list.append({
        "Tweet"             : sample_tweet[:35] + "...",
        "True"              : true_label,
        "Pred"              : prediction,
        "✓?"                : "✅" if is_correct else "❌",
        "RoBERTa P(ironic)" : round(p_roberta, 3),
        "CoT Verdict"       : cot_signal,
        "α (adaptive)"      : round(alpha, 3),
        "Fused P(ironic)"   : round(p_fused, 3),
        "Entropy"           : round(result_dict["entropy"], 3),
        "Gamma"             : round(result_dict["gamma_dyn"], 3),
        "Orig Len"          : len(result_dict["cot_text"].split()),
        "Comp Len"          : len(compressed_text.split()),
        "Critical Tokens"   : str(result_dict["critical_tokens"]),
    })

accuracy_pct = (correct_predictions / NUM_SAMPLES) * 100
skip_ratio   = sum(1 - r["Comp Len"]/r["Orig Len"] for r in results_list) / NUM_SAMPLES

print(f"\n{'='*64}")
print(f"🏆 RDS v2 [{CURRENT_MODE}] ACCURACY : {accuracy_pct:.1f}%  ({correct_predictions}/{NUM_SAMPLES})")
print(f"📉 Avg Token-Skip Ratio         : {skip_ratio:.1%}")
print(f"⚖️  Alpha range used             : [0.35 – 0.75] (entropy-gated)")
print(f"{'='*64}")

df = pd.DataFrame(results_list)
display(df)

🚀 Starting RDS v2 Pipeline in GRADIENT_ONLY mode...
   Upgrades: structured verdict prompt + entropy-adaptive alpha


--- Processing Tweet: @user Can U Help?||More conservatives needed on #TSU + get p... ---
  CoT Length : 61 words
  Crit.Tokens: ['PROG', 'difference', 'groups', 'make', 'doing']
  RoBERTa=0.303 | CoT=0.5 | α=0.550 | Fused=0.392 | Pred=0 | True=0 | ✅

--- Processing Tweet: Just walked in to #Starbucks and asked for a "tall blonde" H... ---
  CoT Length : 92 words
  Crit.Tokens: ['expresses', 'dict', 'nonsensical', 'ironic', 'irony']
  RoBERTa=0.650 | CoT=1.0 | α=0.550 | Fused=0.807 | Pred=1 | True=1 | ✅

--- Processing Tweet: #NOT GONNA WIN... ---
  CoT Length : 96 words
  Crit.Tokens: ['their', 'Step', 'hashtag', 'feelings', 'happy']
  RoBERTa=0.033 | CoT=0.5 | α=0.550 | Fused=0.243 | Pred=0 | True=0 | ✅

--- Processing Tweet: @user He is exactly that sort of person. Weirdo!... ---
  CoT Length : 94 words
  Crit.Tokens: ['Analysis', 'Context', 'general', 'Content', 'ir

,Tweet,True,Pred,✓?,RoBERTa P(ironic),CoT Verdict,α (adaptive),Fused P(ironic),Entropy,Gamma,Orig Len,Comp Len,Critical Tokens
0,@user Can U Help?||More conservativ...,0,0,✅,0.303,0.5,0.55,0.392,0.0,0.5,61,30,"['PROG', 'difference', 'groups', 'make', 'doing']"
1,Just walked in to #Starbucks and as...,1,1,✅,0.650,1.0,0.55,0.807,0.0,0.5,92,46,"['expresses', 'dict', 'nonsensical', 'ironic',..."
2,#NOT GONNA WIN...,0,0,✅,0.033,0.5,0.55,0.243,0.0,0.5,96,48,"['their', 'Step', 'hashtag', 'feelings', 'happy']"
3,@user He is exactly that sort of pe...,0,0,✅,0.209,0.5,0.55,0.340,0.0,0.5,94,47,"['Analysis', 'Context', 'general', 'Content', ..."
4,So much #sarcasm at work mate 10/10...,1,0,❌,0.075,1.0,0.55,0.491,0.0,0.5,79,39,"['work', 'ironic', 'acknowledging', 'atmospher..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,The last day of the year and I will...,0,1,❌,0.133,1.0,0.55,0.523,0.0,0.5,86,43,"['dict', 'contradiction', 'immediately', 'Step..."
96,MB forgot to move the elf on shelf ...,0,1,❌,0.629,1.0,0.55,0.796,0.0,0.5,80,40,"['hearted', 'described', 'disbelief', 'light',..."
97,"@user happy new year, da! Thanks so...",0,0,✅,0.095,0.5,0.55,0.278,0.0,0.5,96,48,"['There', 'meanings', 'hidden', 'Consider', 'S..."
98,WATCH: #Megatron from #Transformers...,0,1,❌,0.515,1.0,0.55,0.733,0.0,0.5,79,39,"['dict', 'indication', 'implying', 'ironic', '..."
